# Git同期ノートブック

Colab等のリモート環境で実験を実行した後、結果ファイル（画像・CSV・HTML等）をリモートブランチにPushするためのノートブック。

**使い方**: 上から順にセルを実行する。

## 1. 設定

In [ ]:
import subprocess
import os
import sys

# === 対象ブランチ名 ===
BRANCH = "feature/exp001-shizuoka-road-network"

# プロジェクトルートの設定
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/geo-laboratory'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))


def run_git(cmd):
    """gitコマンドを実行して結果を表示する"""
    result = subprocess.run(
        cmd, shell=True, cwd=PROJECT_ROOT,
        capture_output=True, text=True
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print(f"stderr: {result.stderr.strip()}")
    return result.returncode


# GitHubのSSHホスト鍵を登録（Colab等で未登録の場合のエラーを防止）
ssh_dir = os.path.expanduser("~/.ssh")
os.makedirs(ssh_dir, exist_ok=True)
known_hosts = os.path.join(ssh_dir, "known_hosts")
result = subprocess.run(
    "ssh-keyscan -t rsa,ecdsa,ed25519 github.com 2>/dev/null",
    shell=True, capture_output=True, text=True
)
if result.stdout.strip():
    with open(known_hosts, "a") as f:
        f.write(result.stdout)
    print("GitHubのSSHホスト鍵を登録しました")

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"対象ブランチ: {BRANCH}")

## 2. Pull（リモートから最新を取得）

In [ ]:
# 現在のブランチを確認し、対象ブランチに切り替え
print("=== 現在のブランチ ===")
run_git("git branch --show-current")

print(f"\n=== ブランチ '{BRANCH}' にチェックアウト ===")
run_git(f"git checkout {BRANCH}")

print("\n=== リモートから最新を取得 ===")
run_git("git pull")

print("\n=== 最新コミット ===")
run_git("git log --oneline -5")

## 3. Push（差分を全てコミットしてプッシュ）

In [ ]:
# 差分の確認
print("=== 変更ファイル一覧 ===")
run_git("git status --short")

# 全差分をステージング
print("\n=== 全差分をステージング ===")
run_git("git add -A")

# ステージング内容の確認
print("\n=== ステージング済みファイル ===")
run_git("git diff --cached --stat")

# コミット
print("\n=== コミット ===")
ret = run_git('git commit -m "docs: 実験結果を追加"')
if ret != 0:
    print("（コミットする変更がありません）")

# プッシュ
print("\n=== プッシュ ===")
run_git("git push")

print("\n=== 完了 ===")
run_git("git log --oneline -3")